# PrakritiAI — 1. Ayurvedic Dataset Generation & EDA

This notebook reproduces the **synthetic Ayurvedic facial-condition dataset** that feeds the PrakritiAI \`DoshaNet\` classifier (canonical implementation: \`src/frontend/src/ml/dataset.ts\`). Everything here mirrors the TypeScript so the numbers you see are the ones the shipped model was trained on.

## Background — Ayurveda & Prakriti
Ayurveda classifies human constitution into three doshas — *Vata*, *Pitta*, *Kapha* — and recognizes 7 *prakriti* types: three **single-dosha** (\`ekadoshaja\`) and four **dual-dosha** (\`dvandvaja\`). Classical texts (Charaka Samhita) describe how each dosha shapes facial features. Because a dual-dosha person exhibits signs of *two* doshas, any observation carries inherent ambiguity — this "label noise" sets a ceiling on achievable accuracy, which is why we evaluate generalization across independent face populations rather than just training loss.

## Feature vocabulary (8 observation categories → 30 one-hot dimensions)
| Feature | Vata-leaning | Pitta-leaning | Kapha-leaning |
|---|---|---|---|
| faceShape | Oval / Oblong | Square / Heart | Round |
| darkCircles | Prominent | Mild–Moderate | Rare (None) |
| puffiness | None | Mild | Significant |
| skinTone | Light, Cool | Fair, Warm | Olive / Deep, Smooth |
| skinMoisture | Dry, Rough | Normal | Oily, Smooth |
| hairTexture | Dry, Frizzy | Fine, Straight | Thick, Oily |
| bodyFrame | Thin, Lean | Medium | Broad, Heavy |
| eyeLook | Small, Dry | Sharp, Piercing | Large, Lustrous |

Each value is sampled *given* a dosha from a probability table grounded in those classical descriptions; the full tables live in \`FEATURE_DEFS\` (defined below, identical to \`dataset.ts\`).

## Contents
1. Reproducible seeded RNG (mulberry32)
2. Feature-definition tables and one-hot encoding
3. The generative model (single vs dual-dosha, dominance probability)
4. Exploratory analyses: class balance, canonical signs, signal-vs-noise


In [ ]:
import numpy as np
from collections import Counter

from prakriti_ml import (
    FEATURE_DEFS, INPUT_SIZE, DOSHA_NAMES, create_rng,
    generate_dataset, encode_conditions,
)


## 1. Reproducible RNG — mulberry32
Deterministic, seed-driven PRNG so every experiment is reproducible (port of \`createRng\` in \`dataset.ts\`).


In [ ]:
rng = create_rng(42)
print("first 5 draws:", [round(rng(), 4) for _ in range(5)])


## 2. Feature definitions & one-hot encoding
Each observation is encoded as a **30-dimensional one-hot vector** (concatenated one-hot for every category). Count them:


In [ ]:
print(f"INPUT_SIZE (one-hot dims) = {INPUT_SIZE}")
print()
for d in FEATURE_DEFS:
    print(f"{d['key']:<14} -> {len(d['values']):>2} values:  {', '.join(d['values'])}")


In [ ]:
# A Vata-leaning face, one-hot encoded
conditions = {
    "faceShape": "Oblong", "darkCircles": "Moderate", "puffiness": "None",
    "skinTone": "Light, Cool", "skinMoisture": "Dry, Rough",
    "hairTexture": "Dry, Frizzy", "bodyFrame": "Thin, Lean",
    "eyeLook": "Small, Dry",
}
vec = encode_conditions(conditions)
print("one-hot vector:")
print(vec.astype(int))
print("active positions:", np.where(vec)[0])
assert vec.shape == (INPUT_SIZE,) and vec.sum() == 8


## 3. The generative model
\`generate_dataset(n, seed, dominance_p, dual_p)\` builds \`n\` phantom faces:

* a **dominant dosha** is drawn with slight class imbalance (Vata 36% / Pitta 32% / Kapha 32%);
* the person is **dual-dosha with probability \`dual_p\`** (default 0.50); a coherent *secondary* dosha is fixed per person;
* for each of the 8 features, the value is sampled from the dominant dosha **with probability \`dominance_p\`** (default 0.90), otherwise from the secondary dosha.

The shipped model uses \`(dominance_p=0.90, dual_p=0.50)\`. The original recipe was \`(0.78, 0.60)\`, which produced a ~7 pt lower accuracy ceiling — shown in notebook 3.


In [ ]:
samples = generate_dataset(4000, seed=42)  # shipped recipe
labels = Counter(s.label for s in samples)
total = len(samples)
print(f"generated {total} phantom faces")
for d in range(3):
    print(f"  {DOSHA_NAMES[d]:<6} {labels[d]:>5}  ({labels[d] / total * 100:5.1f}%)")
print(f"  {'total':<6} {total:>5}  ({100.0:5.1f}%)")


## 4. EDA — canonical signs per dosha
The **modal value** of each feature under each dosha (the "textbook sign" a clinician would look for):

> Note: this simple *modal-agreement* metric is a conservative proxy for learnable signal — the full probability tables (not just each dosha's most likely value) carry additional discriminating information.


In [ ]:
print(f"{'feature':<14}  {'Vata':<20}{'Pitta':<20}{'Kapha':<20}")
for d in FEATURE_DEFS:
    modes = []
    for dosha in range(3):
        modes.append(d["values"][int(np.argmax(d["by_dosha"][dosha]))])
    print(f"{d['key']:<14}  {modes[0]:<20}{modes[1]:<20}{modes[2]:<20}")


In [ ]:
# How often does an observed feature equal its dosha's canonical sign?
# This is the learnable signal; the gap to 100% is the noise a classifier must survive.
agree = []
for s in samples:
    hits = 0
    for d in FEATURE_DEFS:
        observed = d["values"].index(s.conditions[d["key"]])
        canonical = int(np.argmax(d["by_dosha"][s.label]))
        hits += observed == canonical
    agree.append(hits / len(FEATURE_DEFS))
print(f"mean feature agreement with the dominant dosha's canonical sign: {np.mean(agree) * 100:.1f}%")
print(f"range: {np.min(agree) * 100:.0f}% .. {np.max(agree) * 100:.0f}%")


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

counts = [labels[d] for d in range(3)]
axes[0].bar(DOSHA_NAMES, counts, color=["#7c3aed", "#e11d48", "#d97706"])
axes[0].set_title("Class balance (4000 phantom faces)")
axes[0].set_ylabel("count")

fd = FEATURE_DEFS[0]  # faceShape
x = np.arange(3); width = 0.25
for i, v in enumerate(fd["values"]):
    frac = [fd["by_dosha"][d][i] for d in range(3)]
    axes[1].bar(x + (i - 2) * width, frac, width, label=v)
axes[1].set_xticks(x, DOSHA_NAMES)
axes[1].set_title("faceShape value distribution per dosha")
axes[1].set_ylabel("probability")
axes[1].legend(fontsize=7, loc="upper right")

fig.tight_layout(); plt.show()


## 5. Why perfect accuracy is impossible here
A dual-dosha face exposes ~10% of its features from the *secondary* dosha (\`dominance_p = 0.90\`), and even a single dosha's value distribution overlaps its neighbours (e.g. Vata and Kapha can both produce an Oval face). Consequently the Bayes-optimal classifier has a < 100% ceiling. This is *realistic* for a clinical screening task. The practical consequence:

1. we must measure **generalization on unseen populations**, not training loss;
2. the dataset coherence parameters directly set the achievable ceiling — which is exactly the lever the optimization notebook (03) uses.
